<a href="https://colab.research.google.com/github/serdararici/btk-datathon-2026/blob/main/notebooks/v3_nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Datathon 2026 — v3: NLP Feature Engineering
**Goal:** Extract signal from mentor_feedback_text (Turkish)  
**Previous:** v2 → Kaggle MSE 98.83  
**Author:** Serdar Arıcı | **Date:** June 2026

In [13]:
# ============================================================
# SECTION 1: SETUP
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

DATA_PATH    = '/content/drive/MyDrive/Colab Notebooks/btk-datathon-2026/'
DATASET_PATH = DATA_PATH + 'datasets/'
OUTPUT_PATH  = DATA_PATH + 'outputs/'

train = pd.read_csv(DATASET_PATH + 'train.csv', encoding='utf-8-sig')
test  = pd.read_csv(DATASET_PATH + 'test_x.csv', encoding='utf-8-sig')

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Train: (10000, 47)
Test:  (10000, 46)


In [14]:
# ============================================================
# SECTION 2: EXPLORE TEXT DATA
# ============================================================

# Look at high-scoring vs low-scoring feedback to find signal words
print("=== HIGH SCORING STUDENTS (score > 90) ===\n")
high = train[train['career_success_score'] > 90].head(5)
for _, row in high.iterrows():
    print(f"Score: {row['career_success_score']:.1f}")
    print(row['mentor_feedback_text'])
    print()

print("\n=== LOW SCORING STUDENTS (score < 50) ===\n")
low = train[train['career_success_score'] < 50].head(5)
for _, row in low.iterrows():
    print(f"Score: {row['career_success_score']:.1f}")
    print(row['mentor_feedback_text'])
    print()

=== HIGH SCORING STUDENTS (score > 90) ===

Score: 92.5
Ürün analizi alanına olan tutkusu ve makine öğrenimindeki yetkinliği ile dikkat çekiyor. Ancak, portföyünde daha fazla derinlik sağlamak ve işbirliği becerilerini geliştirmek, kariyer hedeflerine ulaşmasında faydalı olabilir.

Score: 94.5
Geri dönüşümlerinizi analiz etmekteki yeteneğiniz ve backend geliştirmedeki başarınız dikkatimi çekti. Aynı zamanda, staj deneyimleriniz ve gerçek müşteri projeleri üzerindeki çalışmanız, sizi sektöre daha hazırlıklı hale getiriyor. Ancak, bulut ve DevOps konularında daha fazla bilgi edinmeniz, kariyer hedeflerinize ulaşmanızı kolaylaştırabilir.

Score: 94.0
Frontend geliştirme konusunda yetenekleri dikkat çekici ve yapmış olduğu projelerle bunu kanıtlıyor. Katıldığı hackathonlar ve açık kaynak projeleri, onun gelişim sürecinde önemli birer adım; ancak iletişim ve takım çalışmasına yönelmesi gereken alanlar bulunuyor.

Score: 91.7
Kariyerine yönelik güçlü bir motivasyon sergileyen öğrenci, özelli

In [15]:
# ============================================================
# SECTION 3: TEXT CLEANING
# ============================================================

import re

def clean_text(text):
    """
    Clean Turkish text for NLP processing:
    - Convert to lowercase (Turkish-aware)
    - Remove punctuation and digits
    - Normalize whitespace
    """
    if pd.isnull(text):
        return ""

    # Turkish-aware lowercase (handle İ/I correctly)
    text = text.replace('İ', 'i').replace('I', 'ı')
    text = text.lower()

    # Remove punctuation and digits, keep Turkish letters
    text = re.sub(r'[^a-zçğıöşü\s]', ' ', text)

    # Normalize whitespace (multiple spaces -> single space)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply cleaning to both datasets
train['feedback_clean'] = train['mentor_feedback_text'].apply(clean_text)
test['feedback_clean']  = test['mentor_feedback_text'].apply(clean_text)

# Show before/after
print("BEFORE:", train['mentor_feedback_text'].iloc[0][:100])
print("\nAFTER: ", train['feedback_clean'].iloc[0][:100])

BEFORE: Proje kalitesi ve makine öğrenimi konusundaki uzmanlığı dikkat çekici. SQL becerisi ile birlikte, De

AFTER:  proje kalitesi ve makine öğrenimi konusundaki uzmanlığı dikkat çekici sql becerisi ile birlikte devo


In [16]:
# ============================================================
# SECTION 4: SENTIMENT-BASED FEATURES
# ============================================================

# Word lists built from observing high vs low scoring feedback
# POSITIVE: words indicating strength, achievement, talent
positive_words = [
    'dikkat', 'çekici', 'güçlü', 'başarı', 'başarılı', 'yetkin', 'yetenek',
    'yetenekli', 'uzmanlık', 'mükemmel', 'kanıtlıyor', 'sağlam', 'ileri',
    'kayda', 'değer', 'avantaj', 'aranan', 'kaliteli', 'üstün', 'parlak',
    'etkileyici', 'olumlu', 'güzel', 'yüksek', 'kuvvetli', 'önemli'
]

# NEGATIVE: words indicating need for improvement, weakness
negative_words = [
    'ancak', 'gereken', 'gerekiyor', 'gerekmekte', 'gerekebilir', 'gerekli',
    'eksik', 'zayıf', 'yetersiz', 'geliştirmesi', 'geliştirmek', 'çalışması',
    'destek', 'gelişime', 'gelişmekte', 'sorun', 'düşük', 'daha', 'fazla'
]

# DEVELOPMENT/POTENTIAL: forward-looking words (neutral-ish)
potential_words = [
    'potansiyel', 'potansiyeli', 'umut', 'verici', 'motivasyon',
    'gelişim', 'çaba', 'azim', 'azmi', 'hedef', 'hedefleri'
]

def count_words(text, word_list):
    """Count how many times words from word_list appear in text."""
    words = text.split()
    return sum(1 for w in words if w in word_list)

def create_sentiment_features(df):
    """Create sentiment-based features from cleaned text."""

    # Count positive, negative, potential words
    df['positive_count'] = df['feedback_clean'].apply(lambda x: count_words(x, positive_words))
    df['negative_count'] = df['feedback_clean'].apply(lambda x: count_words(x, negative_words))
    df['potential_count'] = df['feedback_clean'].apply(lambda x: count_words(x, potential_words))

    # Total word count
    df['word_count'] = df['feedback_clean'].apply(lambda x: len(x.split()))

    # Sentiment score: positive minus negative
    df['sentiment_score'] = df['positive_count'] - df['negative_count']

    # Sentiment ratio: positive / (positive + negative + 1)
    df['sentiment_ratio'] = df['positive_count'] / (df['positive_count'] + df['negative_count'] + 1)

    return df

train = create_sentiment_features(train)
test  = create_sentiment_features(test)

# Check correlations with target
sentiment_features = ['positive_count', 'negative_count', 'potential_count',
                      'word_count', 'sentiment_score', 'sentiment_ratio']

print("Sentiment feature correlations with target:")
for feat in sentiment_features:
    corr = train[feat].corr(train['career_success_score'])
    print(f"  {feat:20s}: {corr:.4f}")

Sentiment feature correlations with target:
  positive_count      : 0.3007
  negative_count      : -0.3140
  potential_count     : -0.1121
  word_count          : 0.0511
  sentiment_score     : 0.3981
  sentiment_ratio     : 0.3919


In [17]:
# ============================================================
# SECTION 5: TF-IDF FEATURES
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer

# Turkish stopwords (common words with no sentiment value)
turkish_stopwords = [
    've', 'ile', 'bir', 'bu', 'da', 'de', 'için', 'olan', 'olarak',
    'çok', 'gibi', 'kadar', 'sonra', 'daha', 'en', 'ya', 'veya',
    'ama', 'ise', 'göre', 'her', 'ki', 'mi', 'ne', 'o', 'şu'
]

# TF-IDF: converts text into numerical vectors
# max_features=50: keep only the 50 most informative words
# We fit on TRAIN only (no data leakage), then transform both
tfidf = TfidfVectorizer(
    max_features=50,
    stop_words=turkish_stopwords,
    ngram_range=(1, 2)  # use single words and word pairs
)

# Fit on train, transform both
tfidf_train = tfidf.fit_transform(train['feedback_clean'])
tfidf_test  = tfidf.transform(test['feedback_clean'])

# Convert to dataframes with meaningful column names
tfidf_cols = [f'tfidf_{w}' for w in tfidf.get_feature_names_out()]
tfidf_train_df = pd.DataFrame(tfidf_train.toarray(), columns=tfidf_cols, index=train.index)
tfidf_test_df  = pd.DataFrame(tfidf_test.toarray(),  columns=tfidf_cols, index=test.index)

print(f"TF-IDF features created: {len(tfidf_cols)}")
print(f"Sample features: {tfidf_cols[:10]}")

# Which TF-IDF words correlate most with target?
tfidf_with_target = tfidf_train_df.copy()
tfidf_with_target['target'] = train['career_success_score'].values
tfidf_corr = tfidf_with_target.corr()['target'].drop('target').sort_values(ascending=False)

print("\nTop 10 positive TF-IDF words:")
print(tfidf_corr.head(10))
print("\nTop 10 negative TF-IDF words:")
print(tfidf_corr.tail(10))

TF-IDF features created: 50
Sample features: ['tfidf_alanında', 'tfidf_alanındaki', 'tfidf_ancak', 'tfidf_ancak teknik', 'tfidf_açık', 'tfidf_backend', 'tfidf_becerileri', 'tfidf_becerilerini', 'tfidf_dikkat', 'tfidf_dikkat çekici']

Top 10 positive TF-IDF words:
tfidf_yüksek        0.217452
tfidf_etkileyici    0.154851
tfidf_onu           0.118241
tfidf_güçlü         0.107438
tfidf_backend       0.086350
tfidf_sahip         0.080086
tfidf_oldukça       0.075334
tfidf_açık          0.066897
tfidf_makine        0.056716
tfidf_özellikle     0.052385
Name: target, dtype: float64

Top 10 negative TF-IDF words:
tfidf_çalışması      -0.052064
tfidf_önemli         -0.057594
tfidf_faydalı        -0.061857
tfidf_olacaktır      -0.066138
tfidf_teknik         -0.071831
tfidf_üzerinde       -0.086533
tfidf_pratik         -0.111951
tfidf_ancak teknik   -0.139623
tfidf_fazla          -0.229161
tfidf_ancak          -0.242288
Name: target, dtype: float64


In [18]:
# ============================================================
# SECTION 6: COMBINE ALL NLP FEATURES
# ============================================================

# Sentiment features to add to main dataset
sentiment_cols = ['positive_count', 'negative_count', 'potential_count',
                  'sentiment_score', 'sentiment_ratio']
# word_count dropped (correlation 0.05, not useful)

# Add sentiment features
train_nlp = train[sentiment_cols].copy()
test_nlp  = test[sentiment_cols].copy()

# Add TF-IDF features
train_nlp = pd.concat([train_nlp, tfidf_train_df], axis=1)
test_nlp  = pd.concat([test_nlp,  tfidf_test_df],  axis=1)

print(f"NLP features shape - Train: {train_nlp.shape}")
print(f"NLP features shape - Test:  {test_nlp.shape}")
print(f"\nTotal NLP features: {train_nlp.shape[1]}")
print(f"  - Sentiment: {len(sentiment_cols)}")
print(f"  - TF-IDF:    {tfidf_train_df.shape[1]}")

NLP features shape - Train: (10000, 55)
NLP features shape - Test:  (10000, 55)

Total NLP features: 55
  - Sentiment: 5
  - TF-IDF:    50


In [19]:
# ============================================================
# SECTION 7: FEATURE ENGINEERING (from v2)
# ============================================================

train_fe = train.copy()
test_fe  = test.copy()

def create_features(df):
    """Numeric feature engineering (same as v2)."""
    technical_cols = ['coding_score', 'problem_solving_score', 'data_structures_score',
                      'sql_score', 'machine_learning_score', 'backend_score',
                      'frontend_score', 'cloud_score', 'devops_score']
    df['avg_technical_score'] = df[technical_cols].mean(axis=1)

    soft_cols = ['communication_score', 'teamwork_score',
                 'leadership_score', 'presentation_score']
    df['avg_soft_skill_score'] = df[soft_cols].mean(axis=1)

    df['experience_score'] = (
        df['internship_count'] * 3 + df['real_client_project_count'] * 2 +
        df['freelance_project_count'] * 1 + df['hackathon_count'] * 1
    )
    df['interview_success_rate'] = df['interviews_attended'] / (df['applications_sent'] + 1)
    df['hackathon_efficiency'] = df['hackathon_awards'] / (df['hackathon_count'] + 1)
    df['github_impact'] = df['github_repo_count'] * df['github_avg_stars'].fillna(0)
    df['total_project_count'] = df['real_client_project_count'] + df['freelance_project_count']
    return df

train_fe = create_features(train_fe)
test_fe  = create_features(test_fe)

# Role weighted score
def create_role_weighted_score(df):
    role_weights = {
        'Data Scientist':       {'machine_learning_score': 3, 'problem_solving_score': 2, 'sql_score': 2, 'data_structures_score': 1, 'coding_score': 1},
        'Data Analyst':         {'sql_score': 3, 'problem_solving_score': 2, 'machine_learning_score': 1, 'coding_score': 1, 'frontend_score': 1},
        'Backend Developer':    {'backend_score': 3, 'coding_score': 2, 'data_structures_score': 2, 'sql_score': 1, 'devops_score': 1},
        'Frontend Developer':   {'frontend_score': 3, 'coding_score': 2, 'problem_solving_score': 1, 'backend_score': 1, 'data_structures_score': 1},
        'AI Engineer':          {'machine_learning_score': 3, 'coding_score': 2, 'data_structures_score': 2, 'cloud_score': 1, 'problem_solving_score': 1},
        'MLOps Engineer':       {'devops_score': 3, 'machine_learning_score': 2, 'cloud_score': 2, 'backend_score': 1, 'coding_score': 1},
        'Cloud Engineer':       {'cloud_score': 3, 'devops_score': 2, 'backend_score': 2, 'coding_score': 1, 'problem_solving_score': 1},
        'DevOps Engineer':      {'devops_score': 3, 'cloud_score': 2, 'backend_score': 2, 'coding_score': 1, 'problem_solving_score': 1},
        'Cybersecurity Analyst':{'problem_solving_score': 3, 'coding_score': 2, 'backend_score': 2, 'devops_score': 1, 'data_structures_score': 1},
        'Product Analyst':      {'problem_solving_score': 3, 'sql_score': 2, 'machine_learning_score': 1, 'communication_score': 1, 'frontend_score': 1},
        'Software Developer':   {'coding_score': 3, 'data_structures_score': 2, 'backend_score': 2, 'problem_solving_score': 1, 'frontend_score': 1},
    }
    weighted_scores = []
    for _, row in df.iterrows():
        role = row['target_role']
        if role in role_weights:
            weights = role_weights[role]
            total_weight = sum(weights.values())
            score = sum(row[skill] * w for skill, w in weights.items()) / total_weight
        else:
            score = row['avg_technical_score']
        weighted_scores.append(score)
    df['role_weighted_score'] = weighted_scores
    return df

train_fe = create_role_weighted_score(train_fe)
test_fe  = create_role_weighted_score(test_fe)

print(f"After feature engineering - Train: {train_fe.shape}, Test: {test_fe.shape}")

After feature engineering - Train: (10000, 62), Test: (10000, 61)


In [20]:
# ============================================================
# SECTION 8: PREPROCESSING (from v2)
# ============================================================

train_processed = train_fe.copy()
test_processed  = test_fe.copy()

# Smart imputation: internship duration
duration_median = train_processed.loc[train_processed['internship_count'] > 0, 'internship_duration_months'].median()
for df in [train_processed, test_processed]:
    df.loc[(df['internship_count'] == 0) & (df['internship_duration_months'].isnull()), 'internship_duration_months'] = 0
    df.loc[(df['internship_count'] > 0) & (df['internship_duration_months'].isnull()), 'internship_duration_months'] = duration_median

# Median imputation
numeric_to_impute = ['english_exam_score', 'portfolio_score', 'github_avg_stars',
                     'open_source_contribution_count', 'linkedin_profile_score', 'hr_interview_score']
medians = {col: train_processed[col].median() for col in numeric_to_impute}
train_processed.fillna(medians, inplace=True)
test_processed.fillna(medians, inplace=True)

# Encoding
tier_mapping = {'Tier 1': 4, 'Tier 2': 3, 'Tier 3': 2, 'Tier 4': 1}
train_processed['university_tier'] = train_processed['university_tier'].map(tier_mapping)
test_processed['university_tier']  = test_processed['university_tier'].map(tier_mapping)

# Separate target before OHE
target_col = train_processed['career_success_score'].copy()
train_processed = train_processed.drop(columns=['career_success_score'])

ohe_cols = ['department', 'target_role']
train_processed = pd.get_dummies(train_processed, columns=ohe_cols, dtype=int)
test_processed  = pd.get_dummies(test_processed,  columns=ohe_cols, dtype=int)

train_processed, test_processed = train_processed.align(test_processed, join='left', axis=1, fill_value=0)

# Drop noise + text columns (we already extracted NLP features separately)
noise_cols = ['hobby', 'preferred_social_media_platform', 'student_id',
              'mentor_feedback_text', 'feedback_clean']
for col in noise_cols:
    if col in train_processed.columns:
        train_processed = train_processed.drop(columns=[col])
    if col in test_processed.columns:
        test_processed = test_processed.drop(columns=[col])

print(f"After preprocessing - Train: {train_processed.shape}, Test: {test_processed.shape}")

After preprocessing - Train: (10000, 72), Test: (10000, 72)


In [21]:
# ============================================================
# SECTION 9: VERIFY ALIGNMENT
# ============================================================

print("career_success_score in train:", 'career_success_score' in train_processed.columns)
print("career_success_score in test: ", 'career_success_score' in test_processed.columns)

print("\nIn train but not in test:", set(train_processed.columns) - set(test_processed.columns))
print("In test but not in train:", set(test_processed.columns) - set(train_processed.columns))

career_success_score in train: False
career_success_score in test:  False

In train but not in test: set()
In test but not in train: set()


In [22]:
# ============================================================
# SECTION 10: COMBINE NUMERIC + NLP FEATURES
# ============================================================

# Reset index to ensure clean concatenation
train_processed = train_processed.reset_index(drop=True)
test_processed  = test_processed.reset_index(drop=True)
train_nlp = train_nlp.reset_index(drop=True)
test_nlp  = test_nlp.reset_index(drop=True)

# Add NLP features (sentiment + TF-IDF)
train_final = pd.concat([train_processed, train_nlp], axis=1)
test_final  = pd.concat([test_processed,  test_nlp],  axis=1)

# Add target back to train
train_final['career_success_score'] = target_col.reset_index(drop=True)

print(f"Final shape - Train: {train_final.shape}")
print(f"Final shape - Test:  {test_final.shape}")

# Verify alignment
print("\nIn train but not in test:", set(train_final.columns) - set(test_final.columns))
print("In test but not in train:", set(test_final.columns) - set(train_final.columns))
print(f"\nMissing train: {train_final.isnull().sum().sum()}")
print(f"Missing test:  {test_final.isnull().sum().sum()}")

Final shape - Train: (10000, 128)
Final shape - Test:  (10000, 127)

In train but not in test: {'career_success_score'}
In test but not in train: set()

Missing train: 0
Missing test:  0


In [26]:
# ============================================================
# FIX: Convert to numpy to avoid XGBoost dtype issues
# ============================================================

# Check for problematic columns
problem_cols = []
for col in train_final.columns:
    if hasattr(train_final[col], 'columns'):
        problem_cols.append(col)

if problem_cols:
    print(f"Problematic columns found: {problem_cols}")
else:
    print("No problematic columns found!")

# Force all columns to float
train_final = train_final.copy()
test_final  = test_final.copy()

for col in train_final.columns:
    try:
        train_final[col] = pd.to_numeric(train_final[col], errors='coerce')
    except:
        pass

for col in test_final.columns:
    try:
        test_final[col] = pd.to_numeric(test_final[col], errors='coerce')
    except:
        pass

print(f"Train dtypes unique: {train_final.dtypes.unique()}")
print(f"Missing after fix - Train: {train_final.isnull().sum().sum()}")
print(f"Missing after fix - Test:  {test_final.isnull().sum().sum()}")

Problematic columns found: ['positive_count', 'negative_count', 'potential_count', 'sentiment_score', 'sentiment_ratio', 'positive_count', 'negative_count', 'potential_count', 'sentiment_score', 'sentiment_ratio']
Train dtypes unique: [dtype('int64') dtype('float64')]
Missing after fix - Train: 0
Missing after fix - Test:  0


In [27]:
# ============================================================
# FIX 2: REMOVE DUPLICATE COLUMNS
# ============================================================

# Check for duplicate columns
duplicate_cols = train_final.columns[train_final.columns.duplicated()].tolist()
print(f"Duplicate columns: {duplicate_cols}")

# Keep first occurrence, drop duplicates
train_final = train_final.loc[:, ~train_final.columns.duplicated()]
test_final  = test_final.loc[:, ~test_final.columns.duplicated()]

# Convert all to float64
train_final = train_final.astype(float)
test_final  = test_final.astype(float)

# Verify
print(f"\nTrain shape after fix: {train_final.shape}")
print(f"Test shape after fix:  {test_final.shape}")
print(f"Duplicate cols remaining: {train_final.columns.duplicated().sum()}")
print(f"Dtypes unique: {train_final.dtypes.unique()}")
print(f"Missing train: {train_final.isnull().sum().sum()}")
print(f"Missing test:  {test_final.isnull().sum().sum()}")

Duplicate columns: ['positive_count', 'negative_count', 'potential_count', 'sentiment_score', 'sentiment_ratio']

Train shape after fix: (10000, 123)
Test shape after fix:  (10000, 122)
Duplicate cols remaining: 0
Dtypes unique: [dtype('float64')]
Missing train: 0
Missing test:  0


In [29]:
# ============================================================
# SECTION 11: MODEL - XGBOOST WITH ALL FEATURES
# ============================================================

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Split
X = train_final.drop(columns=['career_success_score'])
y = train_final['career_success_score']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Convert to numpy to avoid XGBoost dtype issues
X_train_np = X_train.to_numpy(dtype=float)
X_val_np   = X_val.to_numpy(dtype=float)
y_train_np = y_train.to_numpy(dtype=float)
y_val_np   = y_val.to_numpy(dtype=float)

# Same parameters as v1 and v2 (fair comparison)
model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_np, y_train_np)
val_preds = model.predict(X_val_np)

val_mse  = mean_squared_error(y_val_np, val_preds)
val_rmse = np.sqrt(val_mse)

print("=" * 45)
print(f"V1 Validation MSE: 89.89  | Kaggle: 99.09")
print(f"V2 Validation MSE: 88.28  | Kaggle: 98.83")
print(f"V3 Validation MSE: {val_mse:.4f} | Kaggle: ?")
print("=" * 45)
print(f"V3 RMSE: {val_rmse:.4f}")

V1 Validation MSE: 89.89  | Kaggle: 99.09
V2 Validation MSE: 88.28  | Kaggle: 98.83
V3 Validation MSE: 86.8354 | Kaggle: ?
V3 RMSE: 9.3185


In [30]:
# ============================================================
# SECTION 12: CREATE SUBMISSION
# ============================================================

# Convert test to numpy
test_np = test_final.drop(columns=['career_success_score'], errors='ignore').to_numpy(dtype=float)

# Predict
test_preds = model.predict(test_np)
test_preds = np.clip(test_preds, 0, 100)

# Load original test for student_id
test_original = pd.read_csv(DATASET_PATH + 'test_x.csv', encoding='utf-8-sig')

submission = pd.DataFrame({
    'student_id': test_original['student_id'],
    'career_success_score': test_preds
})

submission.to_csv(OUTPUT_PATH + 'submission_v3.csv', index=False)

print("Submission created!")
print(submission.head())
print(f"\nPrediction range: {test_preds.min():.2f} to {test_preds.max():.2f}")
print(f"Prediction mean:  {test_preds.mean():.2f}")

Submission created!
   student_id  career_success_score
0  STU_010001             64.367157
1  STU_010002             72.393463
2  STU_010003             71.636246
3  STU_010004             95.352303
4  STU_010005             80.387299

Prediction range: 32.20 to 100.00
Prediction mean:  76.19


## v3 NLP Results

### NLP Features Created
**Sentiment (5 features):**
- positive_count → correlation 0.30
- negative_count → correlation -0.31
- sentiment_score → correlation 0.40 (strongest new feature!)
- sentiment_ratio → correlation 0.39
- potential_count → correlation -0.11 (used in low-scorer feedback)

**TF-IDF (50 features):**
- Strongest positive: tfidf_yüksek (0.22), tfidf_etkileyici (0.15)
- Strongest negative: tfidf_ancak (-0.24), tfidf_fazla (-0.23)
- "ancak" = strongest negative signal in Turkish feedback text
- Bigram "ancak teknik" also captured as negative signal (-0.14)

### Key Findings
- sentiment_score (0.40) = second strongest feature after project_quality_score (0.54)
- "potential" words negatively correlated → mentors use them for struggling students
- TF-IDF confirmed our manual word lists were correct
- Duplicate sentiment columns detected and removed (fix applied)

### Model Performance
| Version | Val MSE | Val RMSE | Kaggle        |
|---------|---------|----------|---------------|
| V1      | 89.89   | 9.48     | 99.089303     |
| V2      | 88.28   | 9.40     | 98.827308     |
| V3      | 86.84   | 9.32     | 97.334411     |

### Total Feature Count
- Numeric + engineered: 72 features
- NLP (sentiment + TF-IDF): 50 features
- Total: 122 features